# Bigtable

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/bigtable.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is Bigtable?

Bigtable is a distributed, highly scalable, and NoSQL database developed by Google.

## How do you design a good Bigtable row key?

A good Bigtable row key supports the query pattern and avoids hotspotting. Bigtable stores rows lexicographically by key, so monotonically increasing keys like raw timestamps can concentrate writes on one tablet. Use prefixes, salting, bucketing, or reversed timestamps when needed.

Example:

```text
customer_id#reverse_timestamp
region#bucket#device_id#timestamp
```

In [ ]:
from datetime import datetime

MAX_TS = 9_999_999_999   # far-future sentinel for reversed timestamps

def row_key(tenant: str, device_id: str, ts: datetime) -> str:
    # salt spreads hot tenants across tablets; reversed timestamp sorts newest first
    bucket = hash((tenant, device_id)) % 16
    reverse_ts = MAX_TS - int(ts.timestamp())
    return f"{bucket:02d}#{tenant}#{device_id}#{reverse_ts:010d}"

The same idea with lexicographic prefixes:

```text
00#acme#dev-00042#9999999999   -> newest first inside one device
00#acme#dev-00042#9999999900
01#globex#dev-00007#9999999998
```

## When should you use Bigtable instead of BigQuery?

Use Bigtable for low-latency operational lookups, time-series access, and high-throughput key-value style workloads. Use BigQuery for analytical SQL, joins, scans, and aggregations. Bigtable is not a general data warehouse; it is optimized for serving rows by key/range.

Example:

+ Bigtable: fetch recent IoT readings for one device in milliseconds
+ BigQuery: calculate monthly usage by region across all devices

Many architectures use both: Bigtable for serving and BigQuery for analytics.

## How does Bigtable handle data storage?

Bigtable stores data in a sparse, distributed, and multi-dimensional sorted map.

## What are the key features of Bigtable?

Some key features of Bigtable include scalability, high performance, fault tolerance, and automatic load balancing.

## How does Bigtable achieve scalability?

Bigtable achieves scalability by partitioning data into tablets, which are distributed across multiple servers.

## What is a tablet in Bigtable?

A tablet is a range of rows in a Bigtable that is stored and managed independently by a single server.

## How does Bigtable ensure high performance?

Bigtable achieves high performance by leveraging in-memory and distributed storage, as well as employing efficient indexing techniques.

## How does Bigtable ensure fault tolerance?

Bigtable maintains multiple replicas of each tablet, ensuring data durability and availability in case of server failures.

## How does Bigtable handle load balancing?

Bigtable automatically redistributes tablets across servers to balance the workload and maintain performance.

## What consistency model does Bigtable provide?

Within a single cluster, Bigtable provides strong consistency: a read sees the result of every completed write. When an instance has clusters in more than one region, replication between them is asynchronous, so data converges across clusters instead of being immediately consistent. App profile routing decides which behavior a client gets: single-cluster routing for strict consistency, multi-cluster routing for lower-latency replicated reads.

## How does Bigtable support structured data?

Bigtable stores data as byte arrays, allowing developers to interpret the data in any structured format they desire.

## What are some typical use cases for Bigtable?

Bigtable is commonly used for storing and analyzing large amounts of time-series data, as well as for serving real-time applications.

## How does Bigtable handle schema changes?

Bigtable is schema-less, meaning that new columns can be added to the table without affecting existing rows.

## Can you explain the architecture of Bigtable?

Bigtable has a distributed architecture with multiple components, including tablets, tablet servers, and a master server for coordination.

## How does Bigtable handle data replication?

Bigtable replicates data across multiple data centers to ensure durability and availability in case of failures.

## What are the advantages of using Bigtable over traditional relational databases?

Some advantages of Bigtable include scalability, high performance, fault tolerance, and the ability to handle unstructured data efficiently.

## Does Bigtable support ACID transactions?

Bigtable gives ACID semantics at the row level: a row mutation, including a conditional read-modify-write, is atomic. Cross-row transactions are not the design center, so model around single-row atomicity instead of expecting relational-style multi-row transactions.

## How can you interact with Bigtable?

Bigtable provides client libraries for different programming languages, such as Java, Python, and Go, to interact with the database.

## What is the recommended way to perform atomic row-level updates in Bigtable?

Bigtable supports atomic row-level updates using conditional mutations, allowing you to update multiple cells within a row atomically.

## How does Bigtable handle access control and security?

Bigtable integrates with Google Cloud IAM (Identity and Access Management) to provide fine-grained access control and security policies.

## Can you describe the data model used in Bigtable?

Bigtable uses a sparse, distributed, and multidimensional sorted map, where data is indexed by a row key, column key, and timestamp.

## How does Big table handle data sharding and distribution?

Bigtable automatically shards data by range partitioning the row keys and distributes tablets across multiple servers.

## What is the maximum size of a row in Bigtable?

Bigtable enforces hard system limits on row key size, row size, and cell size (see the Cloud Bigtable quotas and limits page), and oversized rows hurt latency and write amplification. Design many small rows rather than a few huge ones, and keep large blobs in Cloud Storage with a reference in Bigtable.

## How does Bigtable handle data compression?

Bigtable provides built-in support for data compression, allowing you to save storage space and improve read and write performance.

## Does Bigtable support secondary indexes?

There is no classic relational secondary index, and the traditional answer was to duplicate data under a second row-key pattern. Current releases add asynchronous secondary indexes and continuous materialized views, plus GoogleSQL for Bigtable, so less of that duplication has to be hand-built.

## Can you explain the role of a tablet server in Bigtable?

A tablet server in Bigtable hosts and serves a set of tablets, handling read and write requests for the data within those tablets.

## How does Bigtable handle hotspots?

Splitting helps, but a monotonically increasing row key still funnels writes into one tablet until it splits, so salting or bucketing the key prefix is the real fix. Watch per-tablet load and CPU utilization in Cloud Monitoring, not just table-level throughput.

## Does Bigtable support full-text search capabilities?

Bigtable is not a search engine. For full-text search over this data, export or stream it to Elasticsearch/OpenSearch or Vertex AI Search. Newer Bigtable releases add k-nearest-neighbor vector search through GoogleSQL for Bigtable, which covers similarity search rather than classic text indexing.

## How does Bigtable handle backups and disaster recovery?

Bigtable provides built-in backup and restore functionality, allowing you to create backups and restore data to a specific point in time.

## Can you explain the difference between Bigtable and HBase?

Bigtable and HBase are similar in many ways, as HBase was inspired by Bigtable. The main difference lies in their underlying infrastructure: Bigtable runs on Google's infrastructure, while HBase runs on top of the Hadoop ecosystem.

## What are the considerations for choosing between Bigtable and other databases like Cassandra or MongoDB?

The choice depends on factors such as data volume, query patterns, scalability requirements, and the need for tight integration with other Google Cloud services.

## How does Bigtable handle compaction?

Bigtable performs compaction by periodically merging smaller sorted files into larger ones, reducing storage overhead and improving read performance.

## Can you explain how Bigtable manages garbage collection?

Bigtable uses an automatic garbage collection process to reclaim disk space by removing older versions of data that are no longer needed.

## How does Bigtable handle data locality?

Bigtable provides a mechanism called Colossus locality, which optimizes data placement to minimize network traffic and improve performance.

## What is the role of the Bigtable master server?

The Bigtable master server handles administrative tasks such as tablet assignment, load balancing, and metadata management.

## How does Bigtable handle concurrent read and write requests?

Bigtable uses optimistic concurrency control, where multiple readers and writers can access the same row simultaneously, ensuring consistency during conflicts.

## Does Bigtable provide automatic indexing for faster querying?

Every row is indexed by its row key automatically, and that is the index the storage engine relies on for point lookups and range scans. For access patterns the row key cannot express, use asynchronous secondary indexes, continuous materialized views, or a duplicated row-key layout.

## How does Bigtable handle time-based data, such as event logs?

Bigtable uses a timestamp associated with each cell, allowing you to store and query time-series data efficiently.

## Can you explain the role of a Bloom filter in Bigtable?

A Bloom filter is a probabilistic data structure used by Bigtable to reduce disk I/O by filtering out irrelevant data during read operations.

## What is the role of the Bigtable client library?

The Bigtable client library provides the necessary APIs and interfaces to interact with Bigtable, making it easier to read, write, and manipulate data.

## Does Bigtable support automatic scaling of storage and compute resources?

Yes, Bigtable can automatically scale storage and compute resources based on workload patterns and configuration settings.

## Can you explain the concept of tablet splitting in Bigtable?

Tablet splitting is the process of dividing a tablet into two or more smaller tablets to evenly distribute the data and workload across servers.

## How does Bigtable handle time-travel queries?

Bigtable allows you to retrieve previous versions of data by specifying a timestamp or a time range in your queries.

## Does Bigtable support change data capture (CDC) for real-time data integration?

Bigtable supports change data capture with Change Streams: you enable a change stream on a table and read mutation records from it, usually with Dataflow. Managed templates stream changes to BigQuery or Pub/Sub, which is the modern path for keeping a warehouse, cache, or downstream service in sync with Bigtable.

## Can you explain the role of Bigtable's compression algorithm, Snappy?

Snappy is a fast and efficient compression algorithm used by Bigtable to reduce the size of stored data and improve read and write performance.

## How does Bigtable handle data replication across regions?

Bigtable uses cross-region replication to asynchronously replicate data to multiple regions, ensuring data durability and availability in case of regional failures.

## What is the impact of schema design on Bigtable performance?

```text
few column families, separated by access pattern:
  cf_hot    small, read on every request
  cf_events append-only, expired by GC
  cf_blob   rarely read, large values that could move to GCS instead
```


## Does Bigtable support integration with popular data processing frameworks like Apache Spark or Apache Beam?

Yes, Bigtable integrates with popular data processing frameworks like Apache Spark and Apache Beam, allowing seamless data processing and analysis.

## Can you explain how Bigtable handles data versioning?

Bigtable assigns a unique timestamp to each cell, allowing multiple versions of a cell's data to be stored and retrieved.

## What is the role of Bigtable's memstore?

Bigtable's memstore is an in-memory data structure that temporarily holds recently written data before flushing it to disk.

## How does Bigtable handle schema evolution?

Bigtable is schemaless at the row level: rows can carry different columns, and new column qualifiers appear without a migration. Removing a column means deleting its cells or letting garbage collection expire them. Column families are the fixed part, and changes to them are online operations.

## Does Bigtable provide automatic data expiration?

In [ ]:
from google.cloud.bigtable import column_family

# keep the newest 3 versions and drop anything older than 30 days
rule = column_family.UnionRule(
    column_family.MaxVersionsGCRule(3),
    column_family.MaxAgeGCAgeRule(30 * 24 * 60 * 60),
)
table.column_family("r", gc_rule=rule).create()

```bash
# inspect and change a family's GC policy from the CLI
cbt -project acme-prod -instance acme-bt ls device_readings
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxage=7d
```

## Can you explain how Bigtable handles data encryption?

Bigtable encrypts data at rest with Google-managed keys by default and supports customer-managed encryption keys (CMEK) through Cloud KMS. Data in transit is protected with TLS. Encrypting individual sensitive fields is an application-level concern.

## How does Bigtable handle data access from different regions?

Bigtable routes read and write requests to the closest replica within a region, minimizing network latency for data access.

## Does Bigtable provide support for aggregations and analytics?

Bigtable is optimized for key-based reads and writes, not for scanning aggregates. In order of preference: maintain aggregates at write time (counter families or aggregate cells), use continuous materialized views, query with GoogleSQL for Bigtable, use Data Boost for large read jobs, or query the data from BigQuery. Hadoop or Dataflow is the heavy-lift path, not the first answer.

## Can you explain how Bigtable handles write amplification?

Bigtable minimizes write amplification by buffering and batching smaller writes into larger, more efficient ones before flushing them to disk.

## How does Bigtable handle data locality in a multi-region setup?

Bigtable replicates data across regions, allowing read and write requests to be served from the closest replica, reducing network latency.

## Can you explain how Big table handles data access control on a per-row basis?

IAM grants are at instance and table level, not per row. For row-level restriction, use authorized views (which scope an application to a row-key prefix and a column set) or isolate tenants in separate tables.

## Does Bigtable support automatic data partitioning?

Yes. Bigtable automatically partitions rows into tablets and splits, merges, and rebalances them as data grows or hot spots appear. The part you design is the row key, so that writes spread across tablets instead of concentrating on one range.

## Can you explain the concept of Bigtable's compaction and memtable?

Compaction is the process of merging smaller sorted files into larger ones to improve storage efficiency. Memtable is an in-memory buffer for recent writes before compaction.

## How does Bigtable handle concurrent access to the same row?

Mutations to a single row are atomic, so concurrent read-modify-write operations cannot partially interleave. Bigtable does not expose client-visible row locks: concurrent writes to the same cell are resolved by timestamp, and clients implement optimistic concurrency with conditional mutations (for example, compare-and-set on a version column).

## Can you explain how Bigtable handles garbage collection of older versions of data?

Bigtable periodically identifies and removes older versions of data during the compaction process to reclaim disk space.

## Does Bigtable support data replication within a single region?

Yes, Bigtable supports data replication within a single region to provide higher availability and durability.

## Can you explain the role of Bigtable's tablet placement policy?

Bigtable's tablet placement policy determines how tablets are assigned to tablet servers to ensure load balancing and efficient resource utilization.

## How does Bigtable handle storage growth over time?

Bigtable automatically scales storage capacity as data grows by adding more servers and tablets to accommodate the increased load.

## Does Bigtable provide data snapshot capabilities?

```bash
# create a backup on a cluster and restore it into a new table
gcloud bigtable backups create orders-2026-10-04 \
  --instance=acme-bt --cluster=acme-bt-c1 --table=orders \
  --retention-period=30d

gcloud bigtable instances tables restore \
  --instance=acme-bt --source=projects/acme-prod/instances/acme-bt/clusters/acme-bt-c1/backups/orders-2026-10-04 \
  --destination=orders_restored --async
```

## Can you explain how Bigtable handles storage and retrieval of large objects?

It does not. Bigtable has row and cell size limits and is optimized for many small rows, so large objects belong in Cloud Storage. Store the object there and keep a reference (bucket, path, size, checksum) in Bigtable.

## How does Bigtable handle schema changes without downtime?

Bigtable supports schema changes without downtime by allowing you to add or remove columns without interrupting the ongoing read and write operations.

## Does Bigtable support full-text search capabilities through integrations?

Yes, Bigtable can be integrated with other full-text search engines like Elasticsearch or Apache Lucene for full-text search capabilities.

## Can you explain how Bigtable handles range scans and filters?

Filters are applied server-side, so a query only returns the cells it needs.

## How does Bigtable handle row-level and column-level access control?

IAM handles instance- and table-level permissions; Bigtable has no relational-style row or column grants. Authorized views and separate tables per tenant are the practical ways to narrow what a client can read.

## Can you explain the role of Bigtable's compaction strategy in performance optimization?

Bigtable's compaction strategy determines when and how to merge smaller sorted files into larger ones to optimize storage efficiency and read performance.

## Does Bigtable provide automatic indexing for efficient querying?

Not in the relational sense: every row is indexed by its row key automatically, which is the index the storage engine uses for point lookups and range scans, but there is no automatic secondary indexing. For other access patterns, design one yourself with asynchronous secondary indexes, continuous materialized views, or a duplicated row-key layout.

## Can you explain how Bigtable handles large-scale data migration?

Bigtable provides tools and utilities to facilitate large-scale data migration, allowing you to import/export data efficiently.

## How does Bigtable handle concurrent updates to the same cell from multiple clients?

Writes to the same cell are resolved by timestamp: the newest write wins, and stale writes with older timestamps can be rejected by a conditional mutation. This is why optimistic concurrency with a version column is the standard pattern.

## Does Bigtable provide integration with popular business intelligence tools?

BI tools do not query Bigtable directly. The usual pattern is to query Bigtable from BigQuery or to export/stream data into BigQuery or a warehouse, then point Looker or Tableau at that. Treat Bigtable as the serving store, not the BI source.

## Can you explain the role of Bigtable's Bloom filter in read operations?

Bigtable's Bloom filter is used during read operations to quickly determine whether a requested row or column may exist in a tablet, reducing unnecessary disk I/O.

## How does Bigtable handle data consistency across replicas?

Reads are strongly consistent within a cluster. With clusters in multiple regions, updates propagate asynchronously and cross-cluster reads are eventually consistent. Pick the app profile routing that matches the requirement instead of assuming one model everywhere.

## Can you explain how Bigtable handles data compression and decompression?

Bigtable uses the Snappy compression algorithm to compress data before storing it on disk. Data is decompressed on-the-fly during read operations.

## Does Bigtable support automatic query optimization?

There is no relational optimizer that rescues a bad access pattern: performance comes from row-key design. Bigtable can also be queried with GoogleSQL for Bigtable, which plans those queries, but for analytical SQL at scale the right move is to query Bigtable from BigQuery rather than treat it as a warehouse.

## Can you explain the role of Bigtable's bloom block filter in read operations?

Bigtable's bloom block filter is a probabilistic data structure that helps skip unnecessary disk reads during the lookup process, improving read performance.

## How does Bigtable handle backup and restore operations?

Bigtable provides built-in backup and restore functionality, allowing you to create backups, schedule regular backups, and restore data to a specific point in time.

## Does Bigtable support integration with popular ETL (Extract, Transform, Load) tools?

Yes, Bigtable can integrate with popular ETL tools like Apache Beam, Google Cloud Dataflow, or Apache NiFi for data extraction, transformation, and loading processes.

## Can you explain how Bigtable handles data access control for multi-tenant environments?

Bigtable leverages Google Cloud IAM's multi-tenancy support to enforce fine-grained access control and isolation between tenants.

## How does Bigtable handle data consistency in a multi-region setup?

Replication across clusters is asynchronous, so multi-cluster instances are eventually consistent across regions, while consistency inside a cluster stays strong. App profiles control whether a request goes to a single cluster (consistent reads) or to the nearest replica (lower latency, replicated reads).

## Can you explain how Bigtable handles schema evolution for existing data?

Bigtable allows you to add or remove columns to the schema without affecting existing data. The new schema will be applied to new writes and subsequent read operations.

## Does Bigtable provide support for complex data types like arrays or JSON?

Bigtable stores data as byte arrays, which allows you to store complex data types like arrays or JSON by serializing them into byte representations.

## Can you explain how Bigtable handles access control for different levels of data granularity?

IAM grants apply at instance and table level, and to backups as resources. Row-key-prefix and column granularity comes from authorized views rather than from IAM alone.

## How does Bigtable handle data distribution across different availability zones within a region?

Bigtable automatically distributes data across different availability zones within a region to ensure high availability and fault tolerance.

## Can you explain the role of Bigtable's read-modify-write operation?

Bigtable's read-modify-write operation allows you to read data, modify it, and write it back atomically within a single transaction, ensuring consistency.

## Does Bigtable support integration with machine learning frameworks like TensorFlow or PyTorch?

Bigtable is a common online feature store because lookups are fast and predictable by row key. It is rarely the training data source: training reads large volumes sequentially, which belongs in BigQuery or Parquet on object storage, while the serving path reads features from Bigtable.

## Can you explain the role of Bigtable's mutation operations in write operations?

Bigtable's mutation operations allow you to specify modifications to be applied during write operations, such as inserting or updating data in specific cells.

## How does Bigtable handle access control for different types of operations, such as read, write, or delete?

IAM defines read, write, and admin permissions at instance and table level. Authorized views then narrow an application's data access to a row-key prefix and a column set when table-level access is too broad.

## Can you explain how Bigtable handles data replication across regions in terms of consistency and latency?

Bigtable replicates data asynchronously across regions, which may result in eventual consistency and varying levels of latency between regions.

## How does Bigtable handle data durability and fault tolerance?

Bigtable ensures data durability and fault tolerance through replication, storing multiple replicas of data across different servers and regions.

## Can you explain how Bigtable handles high availability and seamless failover?

Bigtable provides high availability through replication and automatic failover mechanisms, ensuring continuous access to data even in case of server or region failures.

## Does Bigtable support time travel queries with fine-grained control over historical data retrieval?

Yes, Bigtable supports time travel queries, allowing you to retrieve specific versions of data based on timestamps or time ranges.

## Can you explain how Bigtable handles data partitioning and load balancing?

Bigtable partitions data by range partitioning the row keys, and it automatically balances the distribution of tablets across tablet servers to ensure load balancing.

## How does Bigtable handle access control for data in transit?

Bigtable encrypts data in transit using industry-standard encryption protocols, ensuring secure communication between clients and the Bigtable service.

## Can you explain the role of Bigtable's client-side buffering and batching in optimizing write operations?

Bigtable's client-side buffering and batching allow you to group multiple write operations together before sending them to the server, reducing network overhead and improving write performance.

## How does Bigtable handle data replication and failover in a multi-region setup with data consistency requirements?

In a multi-region setup, Bigtable replicates data across regions and provides automatic failover mechanisms to ensure data consistency and high availability, maintaining replicas across regions in sync.

## Real world use cases for Bigtable

### 1. IoT telemetry serving

Millions of devices write readings every few seconds; the application shows the latest readings per device and a downsampled history.

In [ ]:
from google.cloud import bigtable
from google.cloud.bigtable import column_family

client = bigtable.Client(project="acme-prod", admin=True)
table = client.instance("acme-bt").table("device_readings")
table.create()

# expire readings automatically after 7 days
cf = table.column_family("r", gc_rule=column_family.MaxAgeGCAgeRule(7 * 24 * 60 * 60))
cf.create()

# device prefix plus a reversed timestamp so the newest reading sorts first
row = table.direct_row(b"dev-00042#2026-10-04T12:00:00Z")
row.set_cell("r", b"temperature", b"21.4")
row.set_cell("r", b"humidity", b"48")
row.commit()

```bash
# the same operations from the cbt CLI
cbt -project acme-prod -instance acme-bt createtable device_readings
cbt -project acme-prod -instance acme-bt createfamily device_readings r
cbt -project acme-prod -instance acme-bt set device_readings dev-00042#2026-10-04T12:00:00Z r:temperature=21.4
cbt -project acme-prod -instance acme-bt read device_readings prefix=dev-00042#
```

Why Bigtable: millisecond key lookups at very high write rates, with TTL enforced by garbage collection instead of a cleanup job. Salt or bucket the key prefix when one device or tenant dominates traffic.

### 2. User activity and message history

A messaging product stores per-conversation history and a per-user inbox. Reads are prefix scans ordered by time.

In [ ]:
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"conv-7781#",     # newest first because the timestamp is reversed in the key
    end_key=b"conv-7781$",
    limit=50,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
rows.consume_all()

Why Bigtable: a sorted row key gives "the last 50 messages" with a range scan and no secondary index, while appends stay cheap. Model the "all conversations for a user" query as a second row-key layout or an asynchronous secondary index.

### 3. Online feature store for ML serving

A fraud model needs 30-day aggregates per card at prediction time.

```text
row key: card#c-9182#feature#velocity_24h
columns: f:value, f:updated_at
write:   conditional mutation so an older write cannot overwrite a newer value
read:    one row per feature at scoring time (sub-10 ms p99)
train:   export or stream the same features to BigQuery for offline training
```

Why Bigtable: the serving path needs predictable latency at high QPS, which is exactly a row-key lookup. Do not scan Bigtable to build training sets - use BigQuery or Parquet, and keep both paths fed by one feature definition to avoid training/serving skew.

### 4. Real-time bidding and profile lookup

Ad tech keeps a user profile that every bid request reads, and reads must be fast in every region.

```text
row key:  bucket#user_id          (bucket spreads hot users across tablets)
families: profile (small, rarely changing), events (append-only, GC by age)
routing:  multi-cluster app profile so bids read the nearest replica
```

Why Bigtable: high read throughput with replicated low-latency reads. Multi-cluster routing accepts eventual consistency for profile updates, which bidding tolerates, in exchange for serving reads close to the user.

### 5. Operational serving plus BigQuery analytics

The same data serves the application and the analyst. Change Streams feed the analytical copy instead of a dual write.

```text
Bigtable (serving) --Change Streams--> Dataflow --> BigQuery (analytics) --> BI
                          |
                   Data Boost for large read jobs without competing with serving traffic
```

```sql
-- analytical questions run in BigQuery, not against the serving table
SELECT DATE(event_ts) AS ds,
       COUNT(*) AS events,
       APPROX_QUANTILES(latency_ms, 100)[OFFSET(95)] AS p95
FROM analytics.events_enriched
GROUP BY ds
ORDER BY ds;
```

Why Bigtable: the serving table stays optimized for key lookups, analytics runs on a columnar copy, Change Streams remove the fragile dual write, and Data Boost isolates heavy scans from production traffic.